In [ ]:
uv add ultralytics

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Compute capability:",
        torch.cuda.get_device_capability(0),
    )

## 3. Khởi tạo YOLO26 Pose

In [ ]:
from ultralytics import YOLO

pose_model = YOLO("yolo26s-pose.pt")

## 4. Thử trên một frame

In [ ]:
import cv2

sample_path = manifest_df.iloc[0]["video_path"]

capture = cv2.VideoCapture(str(sample_path))
opened, frame = capture.read()
capture.release()

assert opened and frame is not None, (
    f"Không đọc được video: {sample_path}"
)

results = pose_model.predict(
    source=frame,
    device=0,
    imgsz=640,
    conf=0.25,
    half=True,
    verbose=False,
)

result = results[0]

print("Frame:", frame.shape)
print("Số người:", len(result.boxes))

if result.keypoints is not None:
    print(
        "Keypoints:",
        result.keypoints.xy.shape,
    )
    print(
        "Keypoint data:",
        result.keypoints.data.shape,
    )

## 5. Hiển thị kết quả

In [ ]:
import matplotlib.pyplot as plt
import cv2

annotated_frame = result.plot()

plt.figure(figsize=(14, 8))
plt.imshow(
    cv2.cvtColor(
        annotated_frame,
        cv2.COLOR_BGR2RGB,
    )
)
plt.axis("off")
plt.show()

## 6. Chuyển kết quả về hai người Top và Bottom

In [ ]:
import numpy as np


def yolo_result_to_two_players(
    result,
    frame_height,
    keypoint_threshold=0.25,
):
    joints = np.zeros(
        (2, 17, 2),
        dtype=np.float32,
    )

    scores = np.zeros(
        (2, 17),
        dtype=np.float32,
    )

    if (
        result.boxes is None
        or result.keypoints is None
        or len(result.boxes) == 0
    ):
        return joints, scores

    boxes = (
        result.boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    keypoint_data = (
        result.keypoints.data
        .detach()
        .cpu()
        .numpy()
    )

    xy = keypoint_data[:, :, :2]

    if keypoint_data.shape[2] >= 3:
        confidence = keypoint_data[:, :, 2]
    else:
        confidence = np.ones(
            xy.shape[:2],
            dtype=np.float32,
        )

    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    areas = widths * heights

    # Giữ tối đa hai người có bounding box lớn nhất.
    selected = np.argsort(areas)[::-1][:2]

    boxes = boxes[selected]
    xy = xy[selected]
    confidence = confidence[selected]

    center_y = (
        boxes[:, 1] + boxes[:, 3]
    ) / 2

    # Sắp xếp Top trước, Bottom sau.
    order = np.argsort(center_y)

    boxes = boxes[order]
    xy = xy[order]
    confidence = confidence[order]
    center_y = center_y[order]

    if len(xy) == 2:
        joints[0] = xy[0]
        joints[1] = xy[1]

        scores[0] = confidence[0]
        scores[1] = confidence[1]

    elif len(xy) == 1:
        # Chỉ phát hiện một người:
        # đưa vào Top hoặc Bottom theo vị trí dọc.
        player_slot = (
            0
            if center_y[0] < frame_height / 2
            else 1
        )

        joints[player_slot] = xy[0]
        scores[player_slot] = confidence[0]

    invalid = scores < keypoint_threshold
    joints[invalid] = 0.0

    return joints, scores

In [ ]:
joints, scores = yolo_result_to_two_players(
    result,
    frame_height=frame.shape[0],
)

print("Joints:", joints.shape)
print("Scores:", scores.shape)
print("Top valid joints:", np.count_nonzero(scores[0]))
print("Bottom valid joints:", np.count_nonzero(scores[1]))

## 7. Hàm xử lý một video

In [ ]:
def extract_video_joints_yolo26(
    video_path,
    model,
    image_size=640,
    person_confidence=0.25,
    keypoint_threshold=0.25,
):
    video_joints = []
    video_scores = []

    predictions = model.predict(
        source=str(video_path),
        stream=True,
        device=0,
        imgsz=image_size,
        conf=person_confidence,
        half=True,
        verbose=False,
    )

    for result in predictions:
        frame_height = result.orig_shape[0]

        frame_joints, frame_scores = (
            yolo_result_to_two_players(
                result=result,
                frame_height=frame_height,
                keypoint_threshold=(
                    keypoint_threshold
                ),
            )
        )

        video_joints.append(frame_joints)
        video_scores.append(frame_scores)

    if not video_joints:
        return (
            np.empty(
                (0, 2, 17, 2),
                dtype=np.float32,
            ),
            np.empty(
                (0, 2, 17),
                dtype=np.float32,
            ),
        )

    return (
        np.stack(video_joints).astype(
            np.float32
        ),
        np.stack(video_scores).astype(
            np.float32
        ),
    )

## 8. Thử trên một video

In [ ]:
sample_row = manifest_df.iloc[0]

sample_joints, sample_scores = (
    extract_video_joints_yolo26(
        video_path=sample_row["video_path"],
        model=pose_model,
    )
)

print("Video:", sample_row["video_path"])
print("Joints:", sample_joints.shape)
print("Scores:", sample_scores.shape)

valid_frames = np.any(
    sample_joints != 0,
    axis=(1, 2, 3),
)

print(
    "Tỉ lệ frame hợp lệ:",
    valid_frames.mean(),
)

## 9. Benchmark trước khi chạy toàn bộ

In [ ]:
from time import perf_counter

sample_manifest = manifest_df.head(10)

start_time = perf_counter()
total_frames = 0

for row in sample_manifest.itertuples(
    index=False
):
    joints, scores = (
        extract_video_joints_yolo26(
            row.video_path,
            pose_model,
        )
    )

    total_frames += len(joints)

elapsed = perf_counter() - start_time

print("Số video:", len(sample_manifest))
print("Số frame:", total_frames)
print("Thời gian:", round(elapsed, 2), "giây")
print(
    "Tốc độ:",
    round(total_frames / elapsed, 2),
    "FPS",
)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/home/dutai/khacbinh/badminton-stroke"
)

YOLO_JOINTS_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "joints_npy_yolo26s_pose"
)

YOLO_SCORES_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "scores_npy_yolo26s_pose"
)

YOLO_JOINTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

YOLO_SCORES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
np.save(
    YOLO_JOINTS_ROOT
    / f"{sample_row['clip_id']}_joints.npy",
    sample_joints,
)

np.save(
    YOLO_SCORES_ROOT
    / f"{sample_row['clip_id']}_scores.npy",
    sample_scores,
)